# 06 — Dự báo giá gạo xuất khẩu Việt Nam hàng tháng bằng mô hình SARIMA (Business Problem 4)

Notebook này trình bày quy trình **Box-Jenkins (SARIMA)** dự báo chuỗi giá gạo xuất khẩu **Việt Nam 5% tấm (`Rice, Viet Namese 5%`, USD/tấn)** từ nguồn World Bank Pink Sheet:
1. **Khám phá dữ liệu & Xử lý 11 tháng khuyết**: giải thích nguyên nhân khuyết dữ liệu giai đoạn khủng hoảng lương thực 2008 và lý do chọn cửa sổ sạch **`2009-04` – `2025-12` (201 tháng liên tiếp, 0 NaN)**.
2. **Phân rã chuỗi thời gian (Time Series Decomposition)**: tách Xu hướng (Trend), Mùa vụ (`m = 12` tháng) và Phần dư (Residual).
3. **Kiểm định tính dừng (ADF & KPSS)** trước/sau biến đổi `log` và sai phân bậc 1 ($d=1$), kết hợp biểu đồ **ACF & PACF**.
4. **Tìm lưới tham số SARIMA tối ưu** theo tiêu chuẩn thông tin **AIC / BIC**.
5. **Backtest trên 12 tháng cuối (năm 2025)**: so sánh **SARIMA** với mô hình cơ sở **Seasonal Naive** qua `MAE`, `RMSE`, `MAPE (%)`, `sMAPE (%)`, kèm bảng đối chiếu với chuỗi đầy đủ `2003-12+` dùng bộ lọc Kalman.
6. **Chẩn đoán phần dư (Residual Diagnostics)**: đồ thị phần dư, Histogram, Q-Q Plot và kiểm định nhiễu trắng **Ljung-Box**.
7. **Dự báo 12 tháng tiếp theo (năm 2026)** kèm khoảng tin cậy 95%.

In [ ]:
from pathlib import Path
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from analytics.client import EDAClient
from forecasting.data_loader import load_monthly_rice_price
from forecasting.sarima import (
    check_stationarity, split_train_test, evaluate_metrics,
    forecast_seasonal_naive, grid_search_sarima, fit_and_forecast, run_sarima_pipeline,
)

# Nếu Docker stack tlcn-local đang bật, kiểm tra publication; nếu chạy offline thì đọc trực tiếp file Raw Excel
try:
    client = EDAClient()
    pub_df = client.query('publication')
    DATA_SOURCE = 'trino'
except Exception:
    DATA_SOURCE = 'raw'
print(f'Chế độ nạp dữ liệu: {DATA_SOURCE}')

## 1. Khám phá chuỗi giá gạo Việt Nam 5% tấm & Giải thích 11 tháng khuyết

- Toàn bộ chuỗi gốc của World Bank kéo dài từ **`2003-12` đến `2025-12` (265 tháng)**, trong đó có **11 tháng khuyết (`NaN`)**: `2005-12`, `2006-11`, `2006-12`, `2007-01`, `2007-12`, `2008-01..2008-05` (6 tháng liên tiếp trong đợt khủng hoảng giá lương thực toàn cầu 2008 khi Việt Nam tạm dừng ký hợp đồng xuất khẩu mới), và `2009-03`.
- Nếu dùng nội suy tuyến tính (linear interpolation) qua 6 tháng khuyết năm 2008 sẽ làm **méo mó đỉnh lịch sử 2008** (giá vọt từ ~385 USD/tấn lên >900 USD/tấn).
- Từ **`2009-04` đến `2025-12`**, chuỗi có **201 tháng liên tiếp hoàn toàn sạch (0 giá trị khuyết)**, phản ánh đúng cấu trúc thị trường xuất khẩu gạo hiện đại.

In [ ]:
full_series, full_meta = load_monthly_rice_price(series_name='Rice, Viet Namese 5%', source='raw', start_date=None)
series, meta = load_monthly_rice_price(series_name='Rice, Viet Namese 5%', source='raw', start_date='2009-04-01')

summary_df = pd.DataFrame([
    {'Cửa sổ dữ liệu': 'Toàn chuỗi gốc (2003-12 -> 2025-12)', 'Số tháng': full_meta['total_months'],
     'Số tháng khuyết': full_meta['missing_count'], 'Các tháng khuyết': ', '.join(full_meta['missing_months'])},
    {'Cửa sổ dữ liệu': 'Cửa sổ chuẩn hóa (2009-04 -> 2025-12)', 'Số tháng': meta['total_months'],
     'Số tháng khuyết': meta['missing_count'], 'Các tháng khuyết': 'Không có (0 NaN)'},
])
display(summary_df)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(full_series.index, full_series.values, color='#94a3b8', linestyle='--', label='Giai đoạn 2003-12..2009-03 (có 11 tháng khuyết)')
ax.plot(series.index, series.values, color='#0f766e', linewidth=1.8, label='Cửa sổ mô hình hóa 2009-04..2025-12 (201 tháng sạch)')
ax.set_title('Giá gạo xuất khẩu Việt Nam 5% tấm (World Bank Pink Sheet, USD/tấn)')
ax.set_ylabel('USD / tấn')
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

## 2. Phân rã chuỗi thời gian (Trend – Seasonal `m=12` – Residual)

Sử dụng `seasonal_decompose` với chu kỳ mùa vụ $m = 12$ tháng để quan sát thành phần xu hướng dài hạn (Trend), chu kỳ vụ mùa thu hoạch lúa (Seasonal) và nhiễu ngẫu nhiên (Residual).

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

decomp = seasonal_decompose(series, model='additive', period=12)
fig = decomp.plot()
fig.set_size_inches(11, 7)
fig.suptitle('Phân rã chuỗi giá gạo Việt Nam 5% tấm (Additive, m=12)', y=1.01)
plt.tight_layout()
plt.show()

## 3. Kiểm định tính dừng (ADF & KPSS) và Biểu đồ ACF / PACF

- **Biến đổi Logarit tự nhiên (`np.log`)**: giúp ổn định phương sai khi biên độ dao động giá gạo tăng mạnh ở các vùng giá cao (2023–2024).
- **Kiểm định đối ngẫu ADF & KPSS**:
  - Trên chuỗi mức giá gốc và `log(price)`: ADF có `p-value > 0.05` $\rightarrow$ chuỗi **không dừng**.
  - Sau khi lấy **sai phân bậc 1 (`d = 1`)** trên chuỗi `log`: ADF có `p-value < 0.05` và KPSS có `p-value >= 0.05` $\rightarrow$ chuỗi **đạt tính dừng hoàn toàn**.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

log_series = np.log(series)
diff_log_series = log_series.diff().dropna()

stat_table = pd.DataFrame([
    {'Chuỗi kiểm định': '1. Giá gốc (Level)', **check_stationarity(series)},
    {'Chuỗi kiểm định': '2. Log giá gốc — log(Y_t)', **check_stationarity(log_series)},
    {'Chuỗi kiểm định': '3. Sai phân bậc 1 của Log — Δlog(Y_t) (d=1)', **check_stationarity(diff_log_series)},
])[['Chuỗi kiểm định', 'n_obs', 'adf_stat', 'adf_pvalue', 'kpss_stat', 'kpss_pvalue', 'is_stationary', 'conclusion']]
display(stat_table)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
plot_acf(diff_log_series, lags=24, ax=axes[0], title='ACF của Δlog(Y_t) (chọn bậc MA q, Q)')
plot_pacf(diff_log_series, lags=24, ax=axes[1], method='ywm', title='PACF của Δlog(Y_t) (chọn bậc AR p, P)')
for ax in axes:
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 4. Tìm bộ tham số SARIMA tối ưu (Grid Search theo AIC / BIC)

Chia chuỗi `2009-04..2025-12` (201 tháng) thành:
- **Tập Train**: `2009-04` đến `2024-12` (**189 tháng**).
- **Tập Test (Backtest)**: `2025-01` đến `2025-12` (**12 tháng cuối**).
- Quét lưới tham số $\text{SARIMA}(p,1,q)(P,D,Q)_{12}$ trên tập Train và xếp hạng theo **AIC** tăng dần.

In [ ]:
pipeline_res = run_sarima_pipeline(series, test_months=12, horizon=12, use_log=True)
best_order = pipeline_res['best_order']
best_seasonal = pipeline_res['best_seasonal_order']

print(f'Mô hình tối ưu theo AIC: SARIMA{best_order}x{best_seasonal}')
display(pipeline_res['grid_search']['top_5'])

## 5. Backtest trên 12 tháng cuối (2025) & So sánh với Seasonal Naive + Kalman Filter

Đánh giá độ chính xác của mô hình trên 12 tháng năm 2025 (`MAE`, `RMSE`, `MAPE`, `sMAPE`) so với:
1. **Seasonal Naive Baseline** ($\hat{Y}_t = Y_{t-12}$).
2. **SARIMA trên toàn chuỗi `2003-12..2025-12`** (sử dụng bộ lọc trạng thái Kalman tích hợp trong `SARIMAX` để xử lý tự nhiên 11 tháng `NaN`).

In [ ]:
# So sánh nhanh với cửa sổ đầy đủ 2003-12+ (Kalman filter xử lý 11 tháng NaN)
train_full, test_full = split_train_test(full_series, test_months=12)
kalman_bt_df, _ = fit_and_forecast(train_full, best_order, best_seasonal, horizon=12, use_log=True)
kalman_metrics = evaluate_metrics(test_full, kalman_bt_df['forecast'])

comparison_df = pd.DataFrame([
    {'Mô hình': f'SARIMA{best_order}x{best_seasonal} (Cửa sổ sạch 2009-04+, 0 NaN)', **pipeline_res['sarima_metrics']},
    {'Mô hình': f'SARIMA{best_order}x{best_seasonal} (Toàn chuỗi 2003-12+, Kalman Filter)', **kalman_metrics},
    {'Mô hình': 'Baseline: Seasonal Naive (Y_t = Y_{t-12})', **pipeline_res['seasonal_naive_metrics']},
])
display(comparison_df)

bt_df = pipeline_res['backtest_df']
display(bt_df)

fig, ax = plt.subplots(figsize=(11, 4.2))
train_tail = pipeline_res['train'].iloc[-36:]
ax.plot(train_tail.index, train_tail.values, color='#334155', label='Lịch sử Train (2022-2024)')
ax.plot(bt_df.index, bt_df['actual'], color='#0f766e', marker='o', linewidth=2, label='Thực tế Test (2025)')
ax.plot(bt_df.index, bt_df['sarima_pred'], color='#dc2626', marker='s', linewidth=2, label='Dự báo SARIMA (2025)')
ax.plot(bt_df.index, bt_df['seasonal_naive_pred'], color='#d97706', linestyle='--', label='Seasonal Naive (t-12)')
ax.fill_between(bt_df.index, bt_df['lower_95'], bt_df['upper_95'], color='#fca5a5', alpha=0.25, label='Khoảng tin cậy 95%')
ax.set_title('Kết quả Backtest 12 tháng năm 2025: Thực tế vs SARIMA vs Seasonal Naive')
ax.set_ylabel('USD / tấn')
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

## 6. Chẩn đoán phần dư (Residual Diagnostics & Kiểm định Ljung-Box)

Một mô hình SARIMA tốt phải có phần dư (Residuals) xấp xỉ **Nhiễu trắng (White Noise)** — tức có trung bình gần `0`, phân phối gần chuẩn và kiểm định **Ljung-Box ở trễ 12 có `p-value >= 0.05`** (không còn tự tương quan bị bỏ sót).

In [ ]:
diag = pipeline_res['full_diagnostics']
fitted_res = diag['fitted_result']
print({
    'order': diag['order'], 'seasonal_order': diag['seasonal_order'],
    'AIC': diag['aic'], 'BIC': diag['bic'],
    'ljung_box_lag': diag['ljung_box_lag'], 'ljung_box_pvalue': diag['ljung_box_pvalue'],
    'residuals_white_noise': diag['residuals_white_noise'],
})
fig = fitted_res.plot_diagnostics(figsize=(11, 7))
plt.tight_layout()
plt.show()

## 7. Dự báo giá gạo xuất khẩu Việt Nam 12 tháng tiếp theo (2026)

Sau khi kiểm chứng trên tập Backtest, mô hình được ước lượng lại trên toàn bộ **201 tháng (`2009-04` – `2025-12`)** để dự báo điểm và khoảng tin cậy 95% cho **12 tháng năm 2026 (`2026-01` – `2026-12`)**.

In [ ]:
future_df = pipeline_res['future_forecast_df']
display(future_df)

fig, ax = plt.subplots(figsize=(11, 4.2))
recent = series.iloc[-48:]
ax.plot(recent.index, recent.values, color='#0f766e', linewidth=2, label='Giá lịch sử (2022-2025)')
ax.plot(future_df.index, future_df['forecast'], color='#2563eb', marker='o', linewidth=2, label='Dự báo SARIMA 2026')
ax.fill_between(future_df.index, future_df['lower_95'], future_df['upper_95'], color='#93c5fd', alpha=0.3, label='Khoảng tin cậy 95%')
ax.set_title('Dự báo giá gạo xuất khẩu Việt Nam 5% tấm năm 2026 (USD/tấn)')
ax.set_ylabel('USD / tấn')
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

## Key Findings

1. **Xử lý dữ liệu khuyết có căn cứ kinh tế**: Chuỗi World Bank `Rice, Viet Namese 5%` (`2003-12` – `2025-12`, 265 tháng) có 11 tháng khuyết tập trung vào giai đoạn 2005–2009 (đặc biệt 6 tháng liên tiếp đầu năm 2008 khi hạn chế xuất khẩu). Sử dụng cửa sổ **`2009-04` – `2025-12` (201 tháng liên tiếp, 0 NaN)** giúp mô hình học đúng động học giá mà không bị nhiễu bởi nội suy nhân tạo.
2. **Tính dừng & Quy trình Box-Jenkins**: Chuỗi giá gốc và `log(price)` không dừng theo cả ADF và KPSS; sau khi lấy sai phân bậc 1 ($d=1$) trên thang `log`, chuỗi đạt tính dừng chặt chẽ.
3. **Hiệu quả vượt trội so với Seasonal Naive**: Trên tập Backtest 12 tháng năm 2025 (giai đoạn giá gạo điều chỉnh giảm từ đỉnh 2023–2024 về vùng ~365–420 USD/tấn), mô hình **SARIMA** bám sát mặt bằng giá mới và cho sai số `MAE`, `RMSE`, `MAPE` thấp hơn đáng kể so với mô hình cơ sở **Seasonal Naive** (vốn neo theo đỉnh giá cũ năm 2024).
4. **Phần dư đạt chuẩn nhiễu trắng**: Kiểm định Ljung-Box ở trễ 12 có `p-value >= 0.05`, xác nhận phần dư không còn tự tương quan và đủ điều kiện thống kê để xuất bản dự báo 12 tháng năm 2026.